# Long-Term Patterns of Post-Fire Harvest Diverge Among Ownerships in the Pacific West

In this Jupyter Notebook, we'll walk through the process of applying our post-fire harvest (PFH) mapping method to an example fire using the [package](https://github.com/aazuspan/post-fire-harvest-2024) developed for this project.

## Setup

To begin, we'll import required libraries and initialize the Earth Engine API. Make sure you have an [authenticated Earth Engine account]() before continuing. We're also going to use the [geemap]() package to visualize results.

In [1]:
import ee
import geemap.core as geemap
import pfh
from pfh.scripts.config import OTSU_THRESHOLDS

Create a map that we'll use to visualize layers.

In [2]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text'…

Choose a fire to demonstrate the methods. Note that the fire must be from the [MTBS collection](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_MTBS_burned_area_boundaries_v1) or have compatible properties (e.g. `Ig_Date`).

In [3]:
mtbs = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
fire = ee.Feature(
    mtbs.filter(ee.Filter.eq("Event_ID", "CA4099512207820120818")).first()
)

Map.addLayer(fire, {}, "Test fire")
Map.centerObject(fire)

Next, we'll use the `pfh` package to load annual Landsat pairs over 5 years post-fire. We'll also apply PIF matching between each pair to minimize radiometric differences.

In [4]:
image_pairs = pfh.composites.get_landsat_composites(fire, years=5)

Each pair is a `dict` with `start` and `end` images and other metadata. We can add the start image for Y1 and the end image for Y5 to the map to visualize the changes.

In [5]:
s2_vis_params = {"min": 8000, "max": 19500, "bands": ["SWIR2", "NIR", "Red"]}
Map.addLayer(image_pairs[0]["start"], s2_vis_params, "Y1 - start")
Map.addLayer(image_pairs[4]["end"], s2_vis_params, "Y5 - end")

To minimize radiometric differences between images, we'll apply [pseudo-invariant feature (PIF) matching](https://developers.google.com/earth-engine/tutorials/community/pseudo-invariant-feature-matching) to each pair.

In [6]:
matched_pairs = pfh.composites.match_pairs(
    image_pairs,
    percentile=80,
    method="sed",
    bands=["SWIR2", "Red"],
    geometry=fire.geometry(),
)

Now we can collapse the 5-year time series of spectral changes into a single image showing the maximum spectral difference.

In [7]:
maxdiff = pfh.composites.max_difference(matched_pairs)
maxdiff_vis_params = {"min": 0, "max": 2500, "bands": ["SWIR2", "Red", "Red"]}
Map.addLayer(maxdiff, maxdiff_vis_params, "Max difference (SWIR2)")

We'll apply super non-iterative clustering (SNIC) to the maximum difference composite to identify clusters of pixels with similar spectral change and reduce noise in the final harvest maps.

In [8]:
clustered = pfh.spectral.snic_cluster(maxdiff, cluster_bands=ee.List(["SWIR2", "Red"]))

Now we'll apply thresholds in the red and SWIR2 bands to classify harvested and unharvested pixels. We'll use the thresholds pre-calculated for the paper using [Otsu's method](https://en.wikipedia.org/wiki/Otsu%27s_method), since threshold calculation can be a slow process.

In [9]:
thresholds = ee.FeatureCollection(OTSU_THRESHOLDS)
swir2_threshold = (
    thresholds.filter(ee.Filter.eq("band", "SWIR2")).first().getNumber("threshold")
)
red_threshold = (
    thresholds.filter(ee.Filter.eq("band", "Red")).first().getNumber("threshold")
)

harvests = pfh.spectral.classify_harvests(
    maxdiff, bands=["SWIR2", "Red"], thresholds=[swir2_threshold, red_threshold]
)

The resulting harvest map includes both the presence and timing of harvests, identified based on when the maximum spectral change occurred.

In [10]:
harvest_palette = ["#7a0177", "#c51b8a", "#f768a1", "#fbb4b9", "#feebe2"]
harvest_vis_params = {"min": 1, "max": 5, "palette": harvest_palette}
Map.addLayer(harvests.selfMask(), harvest_vis_params, "Harvest (year)")